# Imaging right: dip one sample, then leave it on the right imaging rack

Picks up one sample from its own slot (sample 0 by default), dips, holds and lifts it exactly as the
immersion sweep in `AIsorb_arm.ipynb` does (through `mid_station`, nothing skipped). Then, instead of
putting it back in its own slot, the arm takes it to the right imaging rack (behind the sample racks),
slot 2 from the front screw, found with `imaging_quick_check_right.ipynb`. The metal end is on the left
and the coated end on the right, under the camera.

So that the electrode never comes down onto the camera, the arm does not come straight down onto the
imaging spot. It:

1. goes straight up from `mid_station` and crosses high above the bench, turning the gripper a quarter turn;
2. comes down 7 cm to the left of the imaging spot (towards the metal end, away from the camera) and
   stops there to ask;
3. goes down until the electrode is 2.5 cm above the rack, slides 7 cm to the right so the coated end
   goes in under the camera, and lowers it into the slot;
4. opens the gripper, lifts 2.5 cm, slides back 7 cm to the left, goes straight up, crosses back turning
   back, and waits at `mid_station`.

The sample's own slot stays empty. The coated end hangs off the right of the imaging rack, so the wet
film touches nothing. The release height is the height the arm puts samples down at in the sample
racks (z = 190.6).

**Before running:** sample 0 is in its own slot, the imaging slot is empty, the camera lens is at
least 5 cm above the rack, the red wire on the gripper has slack, and your hand is on the e-stop. Do
not run anything in another notebook that is connected to the arm at the same time.

Run the cells in order. The first time, leave `CONFIRM_BEFORE_PLACING = True`.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='imaging_right', connect_db=False)
arm = rt.arm

In [ ]:
# ===== Dip one sample, then leave it on the RIGHT imaging rack (not back in its own slot) =====
# Pick up -> mid_station -> above the flask -> dip -> hold -> lift -> mid_station   (as in AIsorb_arm.ipynb)
# -> straight up -> across, high above the bench, turning a quarter turn
# -> down beside the imaging spot, shifted towards the metal end, so the electrode stays clear of the camera
# -> low over the rack, slide towards the camera to the imaging spot -> down into the slot -> open the gripper
# -> up a little, slide back away from the camera -> straight up -> across, turning back -> mid_station.
import time

SAMPLE = 0                    # which sample to dip (its usual rack slot)
DIP_SPEED_MM_S = 20.0         # speed going into the liquid
HOLD_S = 1                    # seconds to stay immersed
LIFT_SPEED_MM_S = 20.0        # speed coming out of the liquid

# Right imaging rack, slot 2 from the front screw (metal end on the left, coated end on the right,
# towards the camera), found with imaging_quick_check_right.ipynb: the fingers held the electrode
# at x = 294, y = 488.1. z = 190.6 is the height the arm puts samples down at in the sample racks.
IMAGING_SPOT = [294.0, 488.1, 190.6, 180.0, 0.0, 180.0]
SAFE_Z = 500.0                # travel height between mid_station and the imaging rack
HOVER_MM = 100.0              # where the arm stops to ask, above the spot beside the imaging spot
SLIDE_MM = 70.0               # how far towards the metal end (left) the arm comes down, away from the camera
SLIDE_HEIGHT_MM = 25.0        # height above the imaging spot at which the electrode slides in and out
PLACE_SPEED_MM_S = 10.0       # speed of the slow part near the rack
CONFIRM_BEFORE_PLACING = True # stop above the imaging rack and wait for your yes

cfg = arm.config


def at_speed(mm_s):
    """Default motion settings with the speed swapped to mm_s."""
    if not 0 < mm_s <= 100:
        raise ValueError(f'Speed {mm_s} mm/s must be between 0 and 100')
    return {**cfg.pos_settings_dict['default'], 'speed': mm_s}


def turned_from(rpy_a, rpy_b):
    """Largest angle (degrees) between two orientations given as roll, pitch, yaw."""
    return max(abs(((a - b) + 180) % 360 - 180) for a, b in zip(rpy_a, rpy_b))


mid = list(cfg.pos_dict['mid_station'])
immersed = cfg.pos_dict['flask_contact_immersed']
above_flask = list(immersed[:2]) + [immersed[2] + cfg.hover_offset_dict['flask']] + list(immersed[3:])
img_x, img_y, img_z = IMAGING_SPOT[:3]
img_rpy = IMAGING_SPOT[3:]
side_x = img_x - SLIDE_MM     # beside the imaging spot, towards the metal end (left), away from the camera

# The arm must point straight down (the normal way or turned for this rack) and be at least 5 cm
# above the racks, so the first move can be straight up.
here = arm.get_cartesian_pos()
pointing_down = min(turned_from(here[3:], mid[3:]), turned_from(here[3:], img_rpy)) <= 5
if not pointing_down or here[2] < 240:
    raise SystemExit('The gripper must point straight down and be at least 5 cm above the racks (z >= 240) '
                     'before this cell. The arm did not move.')
at_mid = max(abs(a - b) for a, b in zip(here[:3], mid[:3])) <= 5 and turned_from(here[3:], mid[3:]) <= 5

route_to_mid = [] if at_mid else [
    ('straight up from where the arm is', [here[0], here[1], SAFE_Z, *here[3:]], 30),
    ('across to above mid_station', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('down to mid_station', mid, 30),
]
route_to_imaging = [
    ('straight up from mid_station', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('across, turning a quarter turn', [side_x, img_y, SAFE_Z, *img_rpy], 30),
    ('down, beside the imaging spot', [side_x, img_y, img_z + HOVER_MM, *img_rpy], 20),
]
place = [
    ('down, low over the rack', [side_x, img_y, img_z + SLIDE_HEIGHT_MM, *img_rpy], PLACE_SPEED_MM_S),
    ('slide towards the camera, to the imaging spot', [img_x, img_y, img_z + SLIDE_HEIGHT_MM, *img_rpy], PLACE_SPEED_MM_S),
    ('down into the slot', [img_x, img_y, img_z, *img_rpy], 5),
]
route_back = [
    ('up a little', [img_x, img_y, img_z + SLIDE_HEIGHT_MM, *img_rpy], 5),
    ('slide back, away from the camera', [side_x, img_y, img_z + SLIDE_HEIGHT_MM, *img_rpy], PLACE_SPEED_MM_S),
    ('straight up', [side_x, img_y, SAFE_Z, *img_rpy], 30),
    ('across, turning back', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('down to mid_station', mid, 30),
]


def go(steps):
    for name, pose, speed in steps:
        print('   ->', name)
        arm.move_to_pos(pose, **at_speed(speed))


# Print the plan, then wait for confirmation before moving anything.
for speed in (DIP_SPEED_MM_S, LIFT_SPEED_MM_S, PLACE_SPEED_MM_S):
    at_speed(speed)           # raises if out of range, before anything moves
print(f'Sample {SAMPLE}: dip {DIP_SPEED_MM_S} mm/s, hold {HOLD_S} s, lift {LIFT_SPEED_MM_S} mm/s,')
print(f'then leave it on the right imaging rack at {[round(v, 1) for v in IMAGING_SPOT]}.')
for name, pose, speed in route_to_mid:
    print(f'  first: {name:40s} -> {[round(v, 1) for v in pose]} at {speed} mm/s')
print('Check: imaging slot empty, sample in its own slot, red wire has slack, hand on the e-stop,')
print('and the camera lens at least 5 cm above the rack (the electrode slides in under it).')
if input('Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')

arm.assert_ready()
if route_to_mid:
    print('[0] to mid_station, high above the bench')
    go(route_to_mid)

print(f'[1] pick up sample {SAMPLE}, then to the flask')
arm.pick_up_sample(SAMPLE)
arm.move_to_mid_station()
arm.move_to_flask()

print(f'[2] dip at {DIP_SPEED_MM_S} mm/s')
arm.move_to_pos(immersed, **at_speed(DIP_SPEED_MM_S))
arm.sample_in_flask = True
if HOLD_S > 0:
    print(f'[3] hold {HOLD_S} s')
    time.sleep(HOLD_S)

print(f'[4] lift at {LIFT_SPEED_MM_S} mm/s, then to mid_station')
arm.move_to_pos(above_flask, **at_speed(LIFT_SPEED_MM_S))
arm.sample_in_flask = False
arm.move_to_mid_station()

print('[5] to the right imaging rack')
go(route_to_imaging)
if CONFIRM_BEFORE_PLACING and input('Gripper above the imaging rack, coated end on the right, nothing in the way '
                                    'of sliding in under the camera? Type yes to put the sample down: '
                                    ).strip().lower() != 'yes':
    raise SystemExit('Stopped above the imaging rack, still holding the sample. '
                     'Run the last cell to put it back in its own slot.')
go(place)
arm.open_gripper()
print('[6] sample left on the imaging rack; back to mid_station')
go(route_back)
print(f'Done. Sample {SAMPLE} is on the right imaging rack. Its own slot is empty.')

### Only if you answered no at the stop above the imaging rack

Takes the sample back to its own slot instead: straight up, across high above the bench turning back,
`mid_station`, then into its own slot as usual. It does nothing if the arm is not where the cell above
stops to ask.

In [ ]:
# ===== Only if the cell above stopped above the imaging rack: put the sample back in its own slot =====
# Straight up -> across, high above the bench, turning back -> mid_station -> into its own slot (as usual).
here = arm.get_cartesian_pos()
if max(abs(here[0] - side_x), abs(here[1] - img_y), abs(here[2] - (img_z + HOVER_MM))) > 5:
    raise SystemExit('The arm is not where the cell above stops to ask, so this cell does nothing.')
if input(f'Type yes to take sample {SAMPLE} back to its own slot: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
go([
    ('straight up', [side_x, img_y, SAFE_Z, *img_rpy], 30),
    ('across, turning back', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('down to mid_station', mid, 30),
])
arm.put_sample_back(SAMPLE)
print(f'Sample {SAMPLE} is back in its own slot.')